# Preprocessing — Phase 2

**Instructions :**
1. Sélectionner les **chemins** (dossier EDF, `quality_summary.tsv`, JSON config, dossier de sortie)
2. Ajuster les **paramètres** de prétraitement et les **seuils de rejet**
3. Cliquer **Charger les participants** pour lister les participants et leurs canaux
4. Pour chaque participant : ajuster les **canaux** à inclure, puis cocher les participants à traiter
5. Cliquer **Lancer le prétraitement**

**Structure de sortie :**

```
{dossier_sortie}/
  reports_preprocessing/
    {id}_preprocessing_report.html    ← rapport HTML avec heatmap artefacts
    {id}_epoch_rejection.tsv          ← 1 ligne/epoch : file_id, epoch_idx, stage, reject_flag, flag par méthode
    {id}_rejection_summary.tsv        ← statistiques de rejet par stade et par méthode (par participant)
    global_epoch_rejection.tsv        ← concaténation des epoch_rejection de tous les participants
    global_rejection_by_stage.tsv     ← tableau pivot : 1 ligne/stade, n et % rejeté total + par méthode
  derivatives/
    [sous-dossiers de groupe miroirs du dossier EDF, si présents]
    {id}_all-epo.fif                  ← toutes les epochs + metadata de rejet (MNE)
```

**Note :** Les epochs ne sont PAS supprimées ici. La Phase 2b permettra d'inspecter les epochs rejetées et de valider/corriger le masque avant de sauvegarder un `_clean-epo.fif` final.

In [ ]:
import os
import io
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import mne
import yasa

import ipywidgets as widgets
from IPython.display import display, HTML
from ipyfilechooser import FileChooser
from specparam import SpectralModel

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')


In [ ]:
def drop_suffix_duplicates(raw):
    """Keep only the -0 variant when MNE creates -0/-1 duplicates for repeated channel names."""
    groups = {}
    for ch in raw.ch_names:
        if ch.endswith('-0') or ch.endswith('-1'):
            base = ch.rsplit('-', 1)[0]
            groups.setdefault(base, []).append(ch)
    to_drop = []
    for base, ch_list in groups.items():
        if any(c.endswith('-0') for c in ch_list):
            to_drop.extend(c for c in ch_list if not c.endswith('-0'))
    if to_drop:
        raw.drop_channels(to_drop)
    return raw, to_drop


def adapt_remap_dict_to_suffixes(raw, remap_dict):
    """Handle MNE's -0 suffix when the remap config uses the base channel name."""
    ch_set = set(raw.ch_names)
    new_remap = {}
    for base_label, target in remap_dict.items():
        if base_label in ch_set:
            new_remap[base_label] = target
        elif f'{base_label}-0' in ch_set:
            new_remap[f'{base_label}-0'] = target
    return new_remap


def compute_rejection_masks(epochs_data_uV, sfreq, hypno_epochs, ptp_thresholds,
                             flat_thresh, grad_thresh,
                             psd_freqs, psds_data_uV2, thresh_error, thresh_r2):
    """
    Compute per-(epoch, channel) boolean rejection masks for each method.

    Parameters
    ----------
    epochs_data_uV  : (n_epochs, n_channels, n_times) array, in µV
    sfreq           : float
    hypno_epochs    : (n_epochs,) array of str  e.g. ['W', 'N2', ...]
    ptp_thresholds  : dict  e.g. {'W': 250, 'N1': 250, 'N2': 300, 'N3': 400, 'R': 250}
    flat_thresh     : float µV  — flag epoch×channel if ptp < this value
    grad_thresh     : float µV/sample  — flag if max |diff| > this
    psd_freqs       : (n_freqs,) array or None  — full PSD freq axis (from 0.5 Hz)
    psds_data_uV2   : (n_epochs, n_channels, n_freqs) array in µV²/Hz, or None
    thresh_error    : float  — flag if specparam MAE > this
    thresh_r2       : float  — flag if specparam R² < this

    Returns
    -------
    dict of (n_epochs, n_channels) bool arrays:
        keys: 'amplitude', 'flat', 'gradient', '1f_error', '1f_r2'
    """
    n_epochs, n_channels, _ = epochs_data_uV.shape

    # Peak-to-peak amplitude vs per-stage threshold
    ptp = np.ptp(epochs_data_uV, axis=-1)  # (n_epochs, n_channels)
    ptp_thresh_arr = np.array([
        ptp_thresholds.get(s, 250.0) for s in hypno_epochs
    ])[:, np.newaxis]
    mask_amplitude = ptp > ptp_thresh_arr

    # Flat signal: ptp below flat threshold
    mask_flat = ptp < flat_thresh

    # Gradient: maximum sample-to-sample absolute difference
    grad = np.max(np.abs(np.diff(epochs_data_uV, axis=-1)), axis=-1)  # (n_epochs, n_channels)
    mask_gradient = grad > grad_thresh

    # 1/f fit quality via specparam.
    # PSD was computed from 0.5 Hz; fit is restricted to >=2 Hz to minimise low-frequency
    # artifact influence (slow waves inflate delta band in deep sleep).
    # A full peak model is used (not max_n_peaks=0) so that periodic components (spindles,
    # alpha, etc.) are removed before evaluating the aperiodic fit quality — using
    # max_n_peaks=0 forces all peak power into the aperiodic fit, degrading R2 and causing
    # almost all N2/REM epochs to be rejected.
    mask_1f_error = np.zeros((n_epochs, n_channels), dtype=bool)
    mask_1f_r2    = np.zeros((n_epochs, n_channels), dtype=bool)

    if psds_data_uV2 is not None and psd_freqs is not None:
        freq_mask = psd_freqs >= 2
        freqs_reduced = psd_freqs[freq_mask]
        sp_model = SpectralModel(
            peak_width_limits=[0.5, 20],
            aperiodic_mode='fixed',
            min_peak_height=0.3,
        )
        for ei in range(n_epochs):
            for ci in range(n_channels):
                psd_reduced = psds_data_uV2[ei, ci, freq_mask]
                try:
                    sp_model.fit(freqs_reduced, psd_reduced)
                    error = sp_model.get_metrics('error', 'mae')
                    r2    = sp_model.get_metrics('gof', 'squared')
                    if error > thresh_error:
                        mask_1f_error[ei, ci] = True
                    if r2 < thresh_r2:
                        mask_1f_r2[ei, ci] = True
                except Exception:
                    mask_1f_error[ei, ci] = True
                    mask_1f_r2[ei, ci]    = True

    return {
        'amplitude': mask_amplitude,
        'flat':      mask_flat,
        'gradient':  mask_gradient,
        '1f_error':  mask_1f_error,
        '1f_r2':     mask_1f_r2,
    }


def build_combined_method_matrix(mask_dict):
    """
    Build an (n_epochs, n_channels) integer matrix encoding which method first
    flagged each cell.  0=none 1=amplitude 2=flat 3=gradient 4=1f_error 5=1f_r2 6=multiple
    """
    methods   = ['amplitude', 'flat', 'gradient', '1f_error', '1f_r2']
    code_map  = {name: i + 1 for i, name in enumerate(methods)}
    shape     = next(iter(mask_dict.values())).shape
    combined  = np.zeros(shape, dtype=int)
    for name, code in code_map.items():
        combined[mask_dict[name]] = code
    n_flags = sum(mask_dict[name].astype(int) for name in methods)
    combined[n_flags > 1] = 6  # overwrite with 'multiple'
    return combined


def plot_rejection_heatmap(combined_matrix, ch_names, hypno_epochs, file_id):
    """
    Plot a channels x epochs heatmap coloured by rejection method.
    Top strip: hypnogram as a YASA-style step-line (W at top, N3 at bottom).
    Gray line for all stages; REM highlighted in red.
    Returns a matplotlib Figure.
    """
    COLORS = ['#1c0a3b', '#c0392b', '#2980b9', '#e67e22', '#f1c40f', '#27ae60', '#7b0000']
    LABELS = ['none', 'amplitude', 'flat', 'gradient', '1/f error', '1/f R2', 'multiple']
    cmap   = mcolors.ListedColormap(COLORS)
    bounds = np.arange(-0.5, 7.5, 1)
    norm   = mcolors.BoundaryNorm(bounds, cmap.N)

    # Hypnogram heights: W at top (4), N3 at bottom (0)
    STAGE_H = {'W': 4, 'R': 3, 'N1': 2, 'N2': 1, 'N3': 0}
    hypno_y = np.array([STAGE_H.get(s, -0.5) for s in hypno_epochs])

    n_channels   = len(ch_names)
    n_epochs     = combined_matrix.shape[0]
    pct_rejected = 100.0 * combined_matrix.any(axis=1).mean()

    fig_w = max(8, min(n_epochs / 6, 20))
    fig_h = max(3, 0.45 * n_channels + 2.0)
    fig, axes = plt.subplots(
        2, 1, figsize=(fig_w, fig_h),
        gridspec_kw={'height_ratios': [1.2, n_channels], 'hspace': 0.04}
    )

    # --- Hypnogram strip ---
    ax_hyp = axes[0]
    x_step = np.arange(n_epochs + 1)
    y_step = np.append(hypno_y, hypno_y[-1])

    # Full step line in gray
    ax_hyp.step(x_step, y_step, where='post', color='#555555', linewidth=1.2)

    # REM epochs highlighted in red
    for ei in range(n_epochs):
        if hypno_epochs[ei] == 'R':
            ax_hyp.plot([ei, ei + 1], [STAGE_H['R'], STAGE_H['R']],
                        color='#c0392b', linewidth=2.5, solid_capstyle='butt')

    ax_hyp.set_yticks([0, 1, 2, 3, 4])
    ax_hyp.set_yticklabels(['N3', 'N2', 'N1', 'REM', 'W'], fontsize=7)
    ax_hyp.set_ylim(-0.8, 4.8)
    ax_hyp.set_xlim(0, n_epochs)
    ax_hyp.set_xticks([])
    for sp in ['top', 'right', 'bottom']:
        ax_hyp.spines[sp].set_visible(False)

    # --- Main heatmap ---
    ax = axes[1]
    im = ax.pcolormesh(
        np.arange(n_epochs + 1), np.arange(n_channels + 1),
        combined_matrix.T,  # (n_channels, n_epochs)
        cmap=cmap, norm=norm, rasterized=True
    )
    ax.set_yticks(np.arange(n_channels) + 0.5)
    ax.set_yticklabels(ch_names, fontsize=8)
    ax.set_xlabel('Epoch index', fontsize=9)
    ax.set_xlim(0, n_epochs)
    ax.set_ylim(0, n_channels)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)

    cbar = fig.colorbar(im, ax=ax, orientation='vertical',
                        fraction=0.02, pad=0.01, ticks=np.arange(7))
    cbar.ax.set_yticklabels(LABELS, fontsize=7)

    fig.suptitle(f'{file_id} — Artefacts ({pct_rejected:.0f}% rejected)',
                 fontsize=11, y=1.01)
    fig.tight_layout()
    return fig


def build_rejection_summary(mask_dict, hypno_epochs, file_id, ch_names):
    """Per-stage, per-method rejection counts DataFrame (summary statistics)."""
    methods = list(mask_dict.keys())
    rows    = []
    for stage in ['W', 'N1', 'N2', 'N3', 'R']:
        stage_mask = (hypno_epochs == stage)
        n_total    = int(stage_mask.sum())
        any_flag   = np.zeros(len(hypno_epochs), dtype=bool)
        for name in methods:
            any_flag |= mask_dict[name].any(axis=1)
        n_rej_any = int((any_flag & stage_mask).sum())
        rows.append({
            'file_id': file_id, 'stage': stage, 'method': 'any',
            'n_total': n_total, 'n_rejected': n_rej_any,
            'pct_rejected': round(100 * n_rej_any / n_total, 1) if n_total > 0 else float('nan'),
        })
        for name in methods:
            per_epoch = mask_dict[name].any(axis=1)
            n_rej = int((per_epoch & stage_mask).sum())
            rows.append({
                'file_id': file_id, 'stage': stage, 'method': name,
                'n_total': n_total, 'n_rejected': n_rej,
                'pct_rejected': round(100 * n_rej / n_total, 1) if n_total > 0 else float('nan'),
            })
    return pd.DataFrame(rows)


def build_epoch_rejection_log(mask_dict, hypno_epochs, file_id):
    """Per-epoch rejection log: one row per epoch with epoch-level (any-channel) method flags."""
    methods  = list(mask_dict.keys())
    n_epochs = len(hypno_epochs)
    rows = []
    for ei in range(n_epochs):
        row = {
            'file_id':     file_id,
            'epoch_idx':   ei,
            'stage':       hypno_epochs[ei],
            'reject_flag': bool(any(mask_dict[m][ei].any() for m in methods)),
        }
        for m in methods:
            col = 'flag_' + m.replace('/', '').replace(' ', '')
            row[col] = bool(mask_dict[m][ei].any())
        rows.append(row)
    return pd.DataFrame(rows)


def build_global_summary_table(all_rejection_summaries):
    """
    Build a dataset-level pivot table from per-participant rejection summaries.
    One row per sleep stage; columns: n_total, n_rejected, pct_rejected (pooled),
    then n_rej_{method} and pct_rej_{method} for each rejection method.
    """
    methods_list = ['amplitude', 'flat', 'gradient', '1f_error', '1f_r2']
    df_all  = pd.concat(all_rejection_summaries, ignore_index=True)
    rows = []
    for stage in ['W', 'N1', 'N2', 'N3', 'R']:
        stage_data = df_all[df_all['stage'] == stage]
        any_data   = stage_data[stage_data['method'] == 'any']
        n_total    = int(any_data['n_total'].sum())
        n_rej      = int(any_data['n_rejected'].sum())
        n_parts    = int(len(any_data))
        row = {
            'stage':          stage,
            'n_participants': n_parts,
            'n_total':        n_total,
            'n_rejected':     n_rej,
            'pct_rejected':   round(100 * n_rej / n_total, 1) if n_total > 0 else float('nan'),
        }
        for m in methods_list:
            m_data  = stage_data[stage_data['method'] == m]
            n_m_rej = int(m_data['n_rejected'].sum())
            row[f'n_rej_{m}']   = n_m_rej
            row[f'pct_rej_{m}'] = round(100 * n_m_rej / n_total, 1) if n_total > 0 else float('nan')
        rows.append(row)
    return pd.DataFrame(rows)


## 1 — Chemins

In [ ]:
fc_edf = FileChooser()
fc_edf.title = '<b>Dossier EDF :</b>'
fc_edf.show_only_dirs = True

fc_quality = FileChooser(filter_pattern='*.tsv')
fc_quality.title = '<b>quality_summary.tsv</b> (depuis quality_overview) :'

fc_config = FileChooser(filter_pattern='*.json')
fc_config.title = '<b>remap_reref_persubject.json</b> (depuis select&amp;remap_channels_edf) :'

hypno_suffix = widgets.Text(
    value='_Hypnogram_remapped.txt',
    description='Suffixe hypno :',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
hypno_suffix_info = widgets.HTML(value='')


def _detect_hypno_suffixes(chooser):
    """Auto-detect hypnogram .txt suffixes in the EDF folder and populate the widget."""
    try:
        if not chooser.selected:
            hypno_suffix_info.value = ''
            return
        edf_folder = Path(chooser.selected)
        edf_files  = sorted(edf_folder.rglob('*.edf'))
        if not edf_files:
            hypno_suffix_info.value = (
                '<small style="color:#888;">Aucun fichier EDF trouvé dans ce dossier.</small>'
            )
            return
        n_total = len(edf_files)
        all_txt = list(edf_folder.rglob('*.txt'))
        suffix_counts = {}
        for edf in edf_files:
            for txt in all_txt:
                if txt.name.startswith(edf.stem):
                    suffix = txt.name[len(edf.stem):]
                    suffix_counts[suffix] = suffix_counts.get(suffix, 0) + 1
        if not suffix_counts:
            hypno_suffix_info.value = (
                '<small style="color:#e67e00;">Aucun hypnogramme .txt détecté — '
                'vérifiez que les fichiers sont présents ou ajustez le suffixe manuellement.</small>'
            )
            return
        best_suffix, best_count = max(suffix_counts.items(), key=lambda x: (x[1], len(x[0])))
        hypno_suffix.value = best_suffix
        parts = [f'<b>{s}</b>&nbsp;(×{c})' for s, c in sorted(suffix_counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if best_count == n_total else '#e67e00'
        hypno_suffix_info.value = (
            f'<small style="color:{color};">Détectés: &nbsp;{"&nbsp;·&nbsp;".join(parts)}'
            f'&nbsp;— {best_count}/{n_total} fichiers correspondants</small>'
        )
    except Exception as e:
        hypno_suffix_info.value = (
            f'<small style="color:#c0392b;">Erreur détection hypnogrammes : {e}</small>'
        )


fc_edf.register_callback(_detect_hypno_suffixes)

fc_out = FileChooser()
fc_out.title = '<b>Dossier de sortie</b> (recevra <em>reports_preprocessing/</em> et <em>derivatives/</em>) :'
fc_out.show_only_dirs = True

display(
    fc_edf,
    hypno_suffix,
    hypno_suffix_info,
    fc_quality,
    fc_config,
    fc_out,
)

## 2 — Paramètres de prétraitement et seuils de rejet

In [ ]:
# ---- Resampling ----
cb_resample = widgets.Checkbox(
    value=False, description='Activer le resampling',
    style={'description_width': 'initial'}
)
txt_target_freq = widgets.BoundedIntText(
    value=256, min=1, max=10000, step=1,
    description='Fréquence cible (Hz) :',
    style={'description_width': '180px'},
    layout=widgets.Layout(width='320px', display='none')
)

def _toggle_resample(change):
    txt_target_freq.layout.display = '' if change['new'] else 'none'

cb_resample.observe(_toggle_resample, names='value')

# ---- Filter ----
cb_filter = widgets.Checkbox(
    value=True, description='Activer le filtre passe-bande',
    style={'description_width': 'initial'}
)
txt_l_freq = widgets.BoundedFloatText(
    value=0.1, min=0.0, max=100.0, step=0.05,
    description='l_freq (Hz) :',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='260px')
)
txt_h_freq = widgets.BoundedFloatText(
    value=40.0, min=1.0, max=500.0, step=1.0,
    description='h_freq (Hz) :',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='260px')
)

# ---- Epoch rejection thresholds ----
_ws = {'description_width': '200px'}
_wl = widgets.Layout(width='360px')

txt_thresh_W  = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0,
    description='Seuil amplitude W (µV) :', style=_ws, layout=_wl)
txt_thresh_N1 = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0,
    description='Seuil amplitude N1 (µV) :', style=_ws, layout=_wl)
txt_thresh_N2 = widgets.BoundedFloatText(value=300.0, min=1.0, max=5000.0, step=10.0,
    description='Seuil amplitude N2 (µV) :', style=_ws, layout=_wl)
txt_thresh_N3 = widgets.BoundedFloatText(value=300.0, min=1.0, max=5000.0, step=10.0,
    description='Seuil amplitude N3 (µV) :', style=_ws, layout=_wl)
txt_thresh_R  = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0,
    description='Seuil amplitude REM (µV) :', style=_ws, layout=_wl)
txt_flat      = widgets.BoundedFloatText(value=1.0, min=0.01, max=100.0, step=0.1,
    description='Signal plat < (µV) :', style=_ws, layout=_wl)
txt_grad      = widgets.BoundedFloatText(value=100.0, min=1.0, max=5000.0, step=10.0,
    description='Gradient max (µV/sample) :', style=_ws, layout=_wl)
txt_1f_error  = widgets.BoundedFloatText(value=0.15, min=0.0, max=1.0, step=0.01,
    description='Seuil erreur 1/f > :', style=_ws, layout=_wl)
txt_1f_r2     = widgets.BoundedFloatText(value=0.85, min=0.0, max=1.0, step=0.01,
    description='Seuil R² 1/f < :', style=_ws, layout=_wl)

display(
    HTML('<b>Rééchantillonnage</b>'),
    cb_resample, txt_target_freq,
    HTML('<br><b>Filtre passe-bande</b>'),
    cb_filter, widgets.HBox([txt_l_freq, txt_h_freq]),
    HTML('<br><b>Seuils de rejet des epochs — amplitude peak-to-peak par stade</b>'),
    widgets.HBox([txt_thresh_W, txt_thresh_N1]),
    widgets.HBox([txt_thresh_N2, txt_thresh_N3]),
    txt_thresh_R,
    HTML('<br><b>Signal plat &amp; gradient</b>'),
    widgets.HBox([txt_flat, txt_grad]),
    HTML('<br><b>Qualité du fit 1/f (specparam)</b>'),
    widgets.HBox([txt_1f_error, txt_1f_r2]),
)


## 3 — Sélection des participants et lancement

In [ ]:
participant_ui   = {}   # file_id -> {'cb': Checkbox, 'channels': {ch: Checkbox}}
vbox_participants = widgets.VBox([])

btn_load  = widgets.Button(description='Charger les participants',
                           button_style='info',
                           layout=widgets.Layout(width='240px', height='36px'))
load_info = widgets.HTML(value='')


def on_load_participants(btn):
    btn.disabled = True
    if not fc_quality.selected or not fc_config.selected:
        load_info.value = ('<span style="color:#c0392b;">'
                           'Sélectionner quality_summary.tsv et le JSON config d\'abord.</span>')
        btn.disabled = False
        return
    try:
        df_q = pd.read_csv(fc_quality.selected, sep='\t')
        with open(fc_config.selected, 'r', encoding='utf-8') as _f:
            cfg = json.load(_f)
    except Exception as e:
        load_info.value = f'<span style="color:#c0392b;">Erreur: {e}</span>'
        btn.disabled = False
        return

    if 'exclude' in df_q.columns:
        excl_all = df_q.groupby('file_id')['exclude'].all()
    else:
        excl_all = pd.Series(dtype=bool)

    all_ids = sorted(df_q['file_id'].unique())
    participant_blocks = []
    global participant_ui
    participant_ui = {}

    # [J] Chaque participant est wrappé indépendamment pour ne pas bloquer les autres
    load_errors = []
    for fid in all_ids:
        try:
            fully_excluded = bool(excl_all.get(fid, False))
            not_in_cfg     = fid not in cfg

            fid_rows = df_q[df_q['file_id'] == fid]
            if 'channel' in df_q.columns and 'exclude' in df_q.columns:
                chs_flags = {row['channel']: bool(row['exclude'])
                             for _, row in fid_rows.iterrows()}
            elif 'channel' in df_q.columns:
                chs_flags = {row['channel']: False for _, row in fid_rows.iterrows()}
            else:
                chs_flags = {ch: False for ch in cfg.get(fid, {}).get('remap', {}).keys()}

            n_ch   = len(chs_flags)
            n_excl = sum(1 for excl in chs_flags.values() if excl)
            suffix_label = ''
            if fully_excluded:
                suffix_label = ' (tous canaux exclus)'
            elif not_in_cfg:
                suffix_label = ' (absent du JSON config)'

            p_cb = widgets.Checkbox(
                value=(not fully_excluded and not not_in_cfg),
                description=f'{fid}{suffix_label}  [{n_ch} canaux, {n_excl} exclus]',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='440px')
            )

            ch_cbs = {}
            for ch, excl in chs_flags.items():
                ch_cb = widgets.Checkbox(
                    value=not excl,
                    description=ch,
                    indent=False,
                    style={'description_width': 'initial'},
                    layout=widgets.Layout(width='120px')
                )
                ch_cbs[ch] = ch_cb

            ch_row = widgets.HBox(
                list(ch_cbs.values()),
                layout=widgets.Layout(flex_wrap='wrap', margin='0 0 10px 28px')
            )
            participant_ui[fid] = {'cb': p_cb, 'channels': ch_cbs}
            participant_blocks.append(widgets.VBox([p_cb, ch_row]))
        except Exception as e:
            load_errors.append(str(fid))
            participant_blocks.append(widgets.HTML(
                value=f'<span style="color:#c0392b;">{fid} — erreur de chargement : {e}</span>'
            ))

    vbox_participants.children = participant_blocks
    n_ok = len(all_ids) - len(load_errors)
    if load_errors:
        load_info.value = (
            f'<span style="color:#2e7d32;">{n_ok} participant(s) chargé(s).</span> '
            f'<span style="color:#c0392b;">{len(load_errors)} erreur(s) : '
            f'{", ".join(load_errors)}</span>'
        )
    else:
        load_info.value = f'<span style="color:#2e7d32;">{len(all_ids)} participant(s) trouvé(s).</span>'
    btn.disabled = False


btn_load.on_click(on_load_participants)

# ---- Run button ----
btn_run      = widgets.Button(description='▶  Lancer le prétraitement',
                              button_style='primary',
                              layout=widgets.Layout(width='260px', height='40px'))
progress     = widgets.IntProgress(min=0, max=1, value=0, bar_style='info',
                                   layout=widgets.Layout(width='500px'))
progress_lbl = widgets.Label(value='')
out_run      = widgets.Output()


def run_preprocessing(btn):
    btn.disabled = True
    out_run.clear_output()

    if not all([fc_edf.selected, fc_quality.selected, fc_config.selected, fc_out.selected]):
        with out_run:
            print('ERREUR : Sélectionner tous les chemins requis (EDF, quality_summary, JSON, sortie).')
        btn.disabled = False
        return

    edf_folder = Path(fc_edf.selected)
    out_root   = Path(fc_out.selected)

    try:
        with open(fc_config.selected, 'r', encoding='utf-8') as _f:
            config_dict = json.load(_f)
    except Exception as e:
        with out_run:
            print(f'ERREUR chargement config : {e}')
        btn.disabled = False
        return

    reports_dir = out_root / 'reports_preprocessing'
    try:
        reports_dir.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        with out_run:
            print(f'ERREUR création dossier de sortie : {e}')
        btn.disabled = False
        return

    do_resample      = cb_resample.value
    target_freq      = int(txt_target_freq.value) if do_resample else None
    do_filter        = cb_filter.value
    l_freq_val       = float(txt_l_freq.value)
    h_freq_val       = float(txt_h_freq.value)
    ptp_thresh = {
        'W':  float(txt_thresh_W.value),
        'N1': float(txt_thresh_N1.value),
        'N2': float(txt_thresh_N2.value),
        'N3': float(txt_thresh_N3.value),
        'R':  float(txt_thresh_R.value),
    }
    flat_thresh_val  = float(txt_flat.value)
    grad_thresh_val  = float(txt_grad.value)
    thresh_error_val = float(txt_1f_error.value)
    thresh_r2_val    = float(txt_1f_r2.value)

    selected_ids = [fid for fid, ui in participant_ui.items() if ui['cb'].value]
    if not selected_ids:
        with out_run:
            print('Aucun participant sélectionné.')
        btn.disabled = False
        return

    progress.max   = len(selected_ids)
    progress.value = 0
    all_rejection_summaries = []
    all_epoch_logs          = []
    failed                  = []
    t_start                 = time.time()
    hypno_lookup = {p.name: p for p in edf_folder.rglob('*.txt')}

    for idx, file_id in enumerate(selected_ids):
        progress_lbl.value = f'{file_id}  ({idx + 1}/{len(selected_ids)})'

        try:  # [A] catch-all par participant — toute exception imprévue est capturée ici

            edf_candidates = list(edf_folder.rglob(f'{file_id}.edf'))
            if not edf_candidates:
                with out_run:
                    print(f'[{file_id}] EDF introuvable dans {edf_folder}')
                failed.append({'file_id': file_id, 'reason': 'EDF not found'})
                progress.value += 1
                continue
            edf_path = edf_candidates[0]

            edf_rel   = edf_path.parent.relative_to(edf_folder)
            deriv_dir = out_root / 'derivatives' / edf_rel
            deriv_dir.mkdir(parents=True, exist_ok=True)

            if file_id not in config_dict:
                with out_run:
                    print(f'[{file_id}] Absent du JSON config — ignoré.')
                failed.append({'file_id': file_id, 'reason': 'not in JSON config'})
                progress.value += 1
                continue

            sub_config        = config_dict[file_id]
            ch_ui             = participant_ui[file_id]['channels']
            selected_channels = [ch for ch, cb in ch_ui.items() if cb.value]
            if not selected_channels:
                with out_run:
                    print(f'[{file_id}] Aucun canal sélectionné — ignoré.')
                failed.append({'file_id': file_id, 'reason': 'no channels selected'})
                progress.value += 1
                continue

            # [load] preload=False : on ne lit que l'en-tête pour l'instant.
            # include= utilise les noms d'ORIGINE du JSON (remap.keys()). Passer include= à la
            # LECTURE (et non un pick paresseux après coup) est important : (a) ça exclut
            # d'emblée les canaux hors-montage à fréquence plus élevée (ex. ECG 512 Hz), ce qui
            # évite un AssertionError du lecteur EDF de MNE sur la lecture partielle ET préserve
            # la fréquence native de l'EEG ; (b) ça évite tout dictionnaire de remap inversé.
            try:
                raw = mne.io.read_raw_edf(
                    str(edf_path), preload=False, encoding='latin-1',
                    include=list(sub_config.get('remap', {}).keys()), verbose=False
                )
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Erreur chargement EDF : {e}')
                failed.append({'file_id': file_id, 'reason': f'EDF loading: {e}'})
                progress.value += 1
                continue

            raw, _ = drop_suffix_duplicates(raw)

            # [B] Renommage canaux origine -> remappé — non-fatal en soi, mais la sélection
            # qui suit en dépend (cf. garde-fou 'present' juste après).
            try:
                remap_adapted = adapt_remap_dict_to_suffixes(raw, sub_config['remap'])
                raw.rename_channels(remap_adapted)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Erreur renommage canaux : {e} — canaux non renommés.')

            # selected_channels porte les noms *remappés* (UI / quality_summary), même
            # namespace que raw après le renommage. On retire les canaux désélectionnés AVANT
            # load_data() pour ne lire sur disque que les canaux finalement gardés.
            present = [ch for ch in selected_channels if ch in raw.ch_names]
            if not present:
                with out_run:
                    print(f'[{file_id}] Aucun canal sélectionné présent après renommage — ignoré.')
                failed.append({'file_id': file_id, 'reason': 'no channels after rename'})
                progress.value += 1
                continue
            to_drop = [ch for ch in raw.ch_names if ch not in present]
            if to_drop:
                raw.drop_channels(to_drop)
            try:
                raw.load_data()   # ne lit sur disque que les canaux gardés
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Erreur lecture données EDF : {e}')
                failed.append({'file_id': file_id, 'reason': f'EDF data loading: {e}'})
                progress.value += 1
                continue

            # [C] Resampling — fatal
            if do_resample and target_freq is not None:
                try:
                    raw.resample(target_freq, npad='auto', verbose=False)
                except Exception as e:
                    with out_run:
                        print(f'[{file_id}] Erreur resampling : {e}')
                    failed.append({'file_id': file_id, 'reason': f'resampling: {e}'})
                    progress.value += 1
                    continue

            # [D] Re-référencement — non-fatal (warn + continuer sans re-ref)
            ref_channels = sub_config.get('ref_channels', [])
            try:
                if ref_channels == 'average':
                    raw.set_eeg_reference(ref_channels='average', verbose=False)
                elif isinstance(ref_channels, list) and ref_channels:
                    ref_present = [ch for ch in ref_channels if ch in raw.ch_names]
                    if ref_present:
                        raw.set_eeg_reference(ref_channels=ref_present, verbose=False)
                        raw.drop_channels(ref_present)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Erreur re-référencement : {e} — ignoré.')

            # [E] Filtrage — fatal
            if do_filter:
                try:
                    raw.filter(
                        l_freq=l_freq_val, h_freq=h_freq_val,
                        l_trans_bandwidth='auto', h_trans_bandwidth='auto',
                        filter_length='auto', method='fir', phase='zero-double',
                        fir_window='hamming', fir_design='firwin', verbose=False
                    )
                except Exception as e:
                    with out_run:
                        print(f'[{file_id}] Erreur filtrage : {e}')
                    failed.append({'file_id': file_id, 'reason': f'filtering: {e}'})
                    progress.value += 1
                    continue

            sf = raw.info['sfreq']

            _hypno_name = f'{file_id}{hypno_suffix.value}'
            hypno_path = hypno_lookup.get(_hypno_name, edf_path.parent / _hypno_name)
            if not hypno_path.exists():
                with out_run:
                    print(f'[{file_id}] Hypnogramme introuvable : {hypno_path.name}')
                failed.append({'file_id': file_id, 'reason': f'hypno not found: {hypno_path.name}'})
                progress.value += 1
                continue
            try:
                expert_hypno = np.loadtxt(str(hypno_path), dtype=str).astype('<U10')
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Erreur chargement hypno : {e}')
                failed.append({'file_id': file_id, 'reason': f'hypno loading: {e}'})
                progress.value += 1
                continue

            expected_epochs = int(np.floor(raw.n_times / sf / 30))
            if expected_epochs != len(expert_hypno):
                with out_run:
                    print(f'[{file_id}] Incompatibilité hypno ({len(expert_hypno)}) '
                          f'vs EEG ({expected_epochs} epochs) — ignoré.')
                failed.append({'file_id': file_id, 'reason': 'hypno/EEG length mismatch'})
                progress.value += 1
                continue

            stage_mapping = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
            hypno_vec = np.full(len(expert_hypno), -1, dtype=float)
            for stage, value in stage_mapping.items():
                hypno_vec[expert_hypno == stage] = value

            # [F] Epoching — fatal
            try:
                epochs = mne.make_fixed_length_epochs(raw, duration=30, preload=True, verbose=False)
                epochs = epochs[:len(expert_hypno)]
                epochs.events[:, 2] = hypno_vec
                epochs.event_id     = stage_mapping
                n_epochs = len(epochs)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Erreur création epochs : {e}')
                failed.append({'file_id': file_id, 'reason': f'epoching: {e}'})
                progress.value += 1
                continue

            if n_epochs == 0:
                with out_run:
                    print(f'[{file_id}] Aucune epoch — ignoré.')
                failed.append({'file_id': file_id, 'reason': 'no epochs'})
                progress.value += 1
                continue

            hypno_epochs   = expert_hypno[:n_epochs]
            epochs_data_uV = epochs.get_data() * 1e6  # (n_epochs, n_channels, n_times)

            # PSD — non-fatal (rejet 1/f désactivé si échec)
            try:
                n_per_seg = int(4 * sf)
                n_overlap = int(n_per_seg / 2)
                fmax_psd  = min(30.0, sf / 2 - 0.5)
                psds_obj  = epochs.compute_psd(
                    method='welch', fmin=0.5, fmax=fmax_psd,
                    n_fft=n_per_seg, n_overlap=n_overlap, n_per_seg=n_per_seg,
                    window='hann', verbose=False
                )
                psds_data_uV2 = psds_obj.get_data() * 1e12  # µV²/Hz
                psd_freqs     = psds_obj.freqs
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Avertissement PSD : {e} — rejet 1/f désactivé.')
                psds_data_uV2 = None
                psd_freqs     = None

            # Masques de rejet — fatal
            try:
                mask_dict = compute_rejection_masks(
                    epochs_data_uV, sf, hypno_epochs,
                    ptp_thresh, flat_thresh_val, grad_thresh_val,
                    psd_freqs, psds_data_uV2, thresh_error_val, thresh_r2_val
                )
                combined_matrix = build_combined_method_matrix(mask_dict)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Erreur calcul masques de rejet : {e}')
                failed.append({'file_id': file_id, 'reason': f'rejection masks: {e}'})
                progress.value += 1
                continue

            reject_epoch = np.zeros(n_epochs, dtype=bool)
            for name in mask_dict:
                reject_epoch |= mask_dict[name].any(axis=1)

            method_priority = ['amplitude', 'gradient', 'flat', '1f_error', '1f_r2']
            reject_method   = np.full(n_epochs, 'none', dtype=object)
            for name in reversed(method_priority):
                reject_method[mask_dict[name].any(axis=1)] = name
            n_methods = sum(mask_dict[name].any(axis=1).astype(int) for name in method_priority)
            reject_method[n_methods > 1] = 'multiple'

            meta_rows = []
            for ei in range(n_epochs):
                row = {
                    'epoch_idx':     ei,
                    'stage':         hypno_epochs[ei],
                    'reject_flag':   bool(reject_epoch[ei]),
                    'reject_method': reject_method[ei],
                }
                for name in method_priority:
                    col = 'flag_' + name.replace('/', '').replace(' ', '')
                    row[col] = bool(mask_dict[name][ei].any())
                meta_rows.append(row)
            metadata_df = pd.DataFrame(meta_rows)

            # [G] Sauvegarde .fif — fatal (produit principal)
            try:
                epochs.metadata = metadata_df
                epochs.save(str(deriv_dir / f'{file_id}_all-epo.fif'), overwrite=True, verbose=False)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Erreur sauvegarde .fif : {e}')
                failed.append({'file_id': file_id, 'reason': f'fif save: {e}'})
                progress.value += 1
                continue

            # [G] Sauvegarde epoch log TSV — non-fatal
            try:
                epoch_log = build_epoch_rejection_log(mask_dict, hypno_epochs, file_id)
                epoch_log.to_csv(str(reports_dir / f'{file_id}_epoch_rejection.tsv'), sep='\t', index=False)
                all_epoch_logs.append(epoch_log)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Erreur sauvegarde epoch_rejection.tsv : {e}')

            # [G] Sauvegarde rejection summary TSV — non-fatal
            rej_summary = None
            try:
                rej_summary = build_rejection_summary(mask_dict, hypno_epochs, file_id, epochs.ch_names)
                rej_summary.to_csv(str(reports_dir / f'{file_id}_rejection_summary.tsv'), sep='\t', index=False)
                all_rejection_summaries.append(rej_summary)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Erreur sauvegarde rejection_summary.tsv : {e}')

            n_rejected = int(reject_epoch.sum())
            pct_rej    = 100 * n_rejected / n_epochs if n_epochs > 0 else 0.0

            # [H] Rapport HTML + heatmap — non-fatal (.fif déjà sauvegardé)
            try:
                report = mne.Report(title=f'{file_id} — Preprocessing Phase 2', verbose=False)

                fig_hm = plot_rejection_heatmap(combined_matrix, epochs.ch_names, hypno_epochs, file_id)
                report.add_figure(fig=fig_hm, title='Rejection heatmap', section='Artefacts', image_format='PNG')
                plt.close(fig_hm)

                if rej_summary is not None:
                    rej_any      = rej_summary[rej_summary['method'] == 'any'][
                        ['stage', 'n_total', 'n_rejected', 'pct_rejected']]
                    summary_html = rej_any.to_html(index=False, border=0, justify='left')
                else:
                    summary_html = '<p><em>Résumé par stade indisponible.</em></p>'
                report.add_html(
                    html=(f'<p><b>Total epochs :</b> {n_epochs} | '
                          f'<b>Rejected :</b> {n_rejected} ({pct_rej:.1f}%)</p>'
                          + summary_html),
                    title='Rejection summary', section='Artefacts'
                )

                params_html = '<table style="border-collapse:collapse;font-size:.9em;">'
                if do_resample:
                    params_html += (f'<tr><td style="padding:3px 10px;"><b>Resampling</b></td>'
                                    f'<td style="padding:3px 10px;">{target_freq} Hz</td></tr>')
                else:
                    params_html += ('<tr><td style="padding:3px 10px;"><b>Resampling</b></td>'
                                    '<td style="padding:3px 10px;">non</td></tr>')
                if do_filter:
                    params_html += (f'<tr><td style="padding:3px 10px;"><b>Filtre</b></td>'
                                    f'<td style="padding:3px 10px;">{l_freq_val}–{h_freq_val} Hz FIR zero-double Hamming</td></tr>')
                else:
                    params_html += ('<tr><td style="padding:3px 10px;"><b>Filtre</b></td>'
                                    '<td style="padding:3px 10px;">non</td></tr>')
                params_html += (
                    f'<tr><td style="padding:3px 10px;"><b>Seuils amplitude</b></td>'
                    f'<td style="padding:3px 10px;">W={ptp_thresh["W"]} N1={ptp_thresh["N1"]} '
                    f'N2={ptp_thresh["N2"]} N3={ptp_thresh["N3"]} R={ptp_thresh["R"]} µV</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>Signal plat</b></td>'
                    f'<td style="padding:3px 10px;">&lt; {flat_thresh_val} µV</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>Gradient</b></td>'
                    f'<td style="padding:3px 10px;">&gt; {grad_thresh_val} µV/sample</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>1/f erreur</b></td>'
                    f'<td style="padding:3px 10px;">&gt; {thresh_error_val}</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>1/f R²</b></td>'
                    f'<td style="padding:3px 10px;">&lt; {thresh_r2_val}</td></tr>'
                    '</table>'
                )
                report.save(str(reports_dir / f'{file_id}_preprocessing_report.html'),
                            overwrite=True, open_browser=False, verbose=False)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Erreur génération rapport HTML : {e}')

            edf_rel_str = str(edf_rel) if str(edf_rel) != '.' else ''
            with out_run:
                print(f'[{file_id}]  {n_rejected}/{n_epochs} epochs rejetées ({pct_rej:.1f}%)'
                      + (f'  →  derivatives/{edf_rel_str}' if edf_rel_str else '  →  derivatives/'))

            progress.value = idx + 1

        except Exception as e:  # [A] catch-all par participant
            with out_run:
                print(f'[{file_id}] ERREUR inattendue : {repr(e)}')
            failed.append({'file_id': file_id, 'reason': f'unexpected: {repr(e)}'})
            progress.value += 1

    # ---- Global outputs ---- [K]
    try:
        if all_epoch_logs:
            pd.concat(all_epoch_logs, ignore_index=True).to_csv(
                str(reports_dir / 'global_epoch_rejection.tsv'), sep='\t', index=False
            )
            with out_run:
                print('\nRésumé global (par epoch)  →  global_epoch_rejection.tsv')
    except Exception as e:
        with out_run:
            print(f'⚠ Erreur sauvegarde global_epoch_rejection.tsv : {e}')

    try:
        if all_rejection_summaries:
            df_global_table = build_global_summary_table(all_rejection_summaries)
            df_global_table.to_csv(
                str(reports_dir / 'global_rejection_by_stage.tsv'), sep='\t', index=False
            )
            display_cols = ['stage'] + [c for c in df_global_table.columns if c.startswith('pct_')]
            pct_cols = [c for c in display_cols if c.startswith('pct_')]
            styled = (
                df_global_table[display_cols].style
                .set_table_styles([
                    {'selector': 'th', 'props': [('background', '#555'), ('color', '#fff'),
                                                  ('padding', '4px 12px'), ('text-align', 'left')]},
                    {'selector': 'td', 'props': [('padding', '3px 12px'),
                                                  ('border-bottom', '1px solid #eee')]},
                    {'selector': 'tr:nth-child(even) td', 'props': [('background', '#f8f8f8')]},
                ])
                .format({c: '{:.1f}' for c in pct_cols})
                .hide(axis='index')
            )
            with out_run:
                print('Résumé global (par stade)  →  global_rejection_by_stage.tsv')
                display(styled)
    except Exception as e:
        with out_run:
            print(f'⚠ Erreur génération résumé global par stade : {e}')

    if failed:
        try:
            pd.DataFrame(failed).to_csv(
                str(reports_dir / 'preprocessing_failed.tsv'), sep='\t', index=False
            )
            with out_run:
                print(f'\n{len(failed)} participant(s) en erreur  →  preprocessing_failed.tsv')
        except Exception as e:
            with out_run:
                print(f'\n{len(failed)} participant(s) en erreur. '
                      f'⚠ Erreur sauvegarde preprocessing_failed.tsv : {e}')

    elapsed = time.time() - t_start
    mins, secs = divmod(elapsed, 60)
    elapsed_str = f'{int(mins)} min {secs:.0f} s' if mins >= 1 else f'{secs:.1f} s'
    progress_lbl.value = f'Terminé. ({elapsed_str})'
    btn.disabled = False


btn_run.on_click(run_preprocessing)

display(
    widgets.HBox([btn_load, load_info]),
    vbox_participants,
    HTML('<hr style="margin:16px 0;">'),
    btn_run,
    widgets.HBox([progress, progress_lbl]),
    out_run,
)